# Preparación de entrenamiento · raw → refined → experimentos

Muestreo progresivo 20 → 50 → 100%, con transporte incremental y detecciones pendientes en GPU.
El paquete restaura la versión refinada si existe. Si falta, recupera cachés, valida CPU/GPU y amplía los datos.

Para reanudar en la misma VM, interrumpe una vez y espera el respaldo. No borres el runtime ni sus archivos.
Una sola sesión escribe en `MyDrive/PetTrace/checkpoints/preparation`. Este notebook aún no contiene resultados.


## 1. Entorno y protocolo

Instala el wheel local **0.2.5.dev3**, `SHA256SUMS` y `requirements-e3-runtime.txt` en una sesión nueva.
Es una compilación de desarrollo con procedencia registrada, no una release publicada.
No actualices paquetes durante una preparación activa.

Fuentes: [Dogs of the World v1](https://www.kaggle.com/datasets/lextoumbourou/dogs-world)
y [LCW v1](https://www.kaggle.com/datasets/dseidli/lcwlabeled-cats-in-the-wild).
El porcentaje se calcula sobre las fotos originales antes de filtrar, con semilla 42 y muestras anidadas.
La selección, mínimo de cuatro fotos distintas, deduplicación y cuarentena son decisiones de esta investigación.
Las etiquetas de origen no equivalen a identidades verificadas. E1/E2 permanecen congelados.


In [ ]:
# Upload the built wheel, SHA256SUMS and requirements-e3-runtime.txt to a fresh Colab runtime.
import hashlib
import subprocess
import sys
from importlib import metadata
from pathlib import Path

if sys.version_info[:2] != (3, 13) or sys.platform != "linux":
    raise RuntimeError("This E3 Colab profile requires Linux and Python 3.13")
requirements = next((p for p in (Path("requirements-e3-runtime.txt"), Path("requirements/e3-runtime.txt")) if p.is_file()), None)
wheel = Path("pettrace_ml-0.2.5.dev3-py3-none-any.whl")
checksums = Path("SHA256SUMS")
if requirements is None or not wheel.is_file() or not checksums.is_file():
    raise FileNotFoundError("Place the v0.2.5.dev3 wheel, SHA256SUMS and requirements-e3-runtime.txt beside this notebook in the runtime")
if hashlib.sha256(requirements.read_bytes()).hexdigest() != "906d040f5ea4881e4750b553908afd65cf3bc88a64d8e741951e9f9ffd64aa3a":
    raise ValueError("E3 requirements checksum mismatch")
expected = {name: digest for digest, name in (line.split() for line in checksums.read_text().splitlines() if line.strip())}
if hashlib.sha256(wheel.read_bytes()).hexdigest() != expected.get(wheel.name):
    raise ValueError("E3 wheel checksum mismatch")
loaded = {name.split(".")[0] for name in sys.modules}
before = {dist: metadata.version(dist) for module, distributions in metadata.packages_distributions().items()
          if module in loaded for dist in distributions}
install_log = Path("e3-install.log")
with install_log.open("w") as log:
    for args in (["--require-hashes", "-r", str(requirements)], ["--no-deps", str(wheel)]):
        result = subprocess.run([sys.executable, "-m", "pip", "install", *args], stdout=log, stderr=subprocess.STDOUT)
        if result.returncode:
            raise RuntimeError(f"Installation failed; inspect {install_log}")
_E3_RESTART_REQUIRED = sorted(set(globals().get("_E3_RESTART_REQUIRED", [])) |
                              {dist for dist, old in before.items() if metadata.version(dist) != old})
print("Restart this session, then continue at verification." if _E3_RESTART_REQUIRED else "Installed; continue at verification.")


In [ ]:
if globals().get("_E3_RESTART_REQUIRED"):
    raise RuntimeError("Restart before loading the new package")
import os
import torch
import pettrace
from pettrace.tracking import source_identity
from pettrace.data.workflow import TrainingPreparation

if pettrace.__version__ != "0.2.5.dev3":
    raise RuntimeError("Install the v0.2.5.dev3 wheel and restart before continuing")
code_identity = source_identity()
if code_identity["build"].get("version") != "0.2.5.dev3":
    raise RuntimeError("Use the development wheel to retain its commit and lockfile provenance")
print("Package:", pettrace.__version__, "· commit:", code_identity["git_sha"])
print("Source SHA256:", code_identity["source_sha256"])
print("Working tree changes in build:", code_identity["git_dirty"])


## 2. Recuperar la base refinada

Las versiones existentes son inmutables: se cargan y verifican; no se regeneran con otras reglas.
Si una versión falta, se reutilizan las cachés de adquisición y YOLO de preparaciones anteriores.
Los experimentos solo necesitan la raíz `prepared`, que contiene imágenes relativas, manifiesto,
splits, auditoría y hashes. No necesitan las fuentes ni los ZIP.


In [ ]:
import json
from pathlib import Path
from time import perf_counter
from IPython.display import Image, Markdown, display

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
SOURCE_SAMPLE_PCT = 50  # 20, 50 o 100 de las fotos originales, antes de filtrar.
PUBLISH_TO_HF = False
preparation = TrainingPreparation(
    drive_root="/content/drive/MyDrive/PetTrace", work_dir="/content",
    source_percent=SOURCE_SAMPLE_PCT, workers=4, screening_provider="cuda")
prepared = preparation.restore()
print(f"Refined {preparation.version}: " + ("verificada; limpieza omitida." if prepared else "pendiente; se recuperará la caché."))


## 3. Ampliar raw y limpiar solo si falta la versión

La lógica vive en `pettrace.data.workflow.TrainingPreparation`: restaurar, preparar y publicar son métodos
reutilizables fuera del notebook. Los ZIP solo se copian si la adquisición está incompleta; la extracción
incremental usa cuatro trabajadores. Las detecciones CPU compatibles se conservan; las nuevas CUDA tienen caché separada.

Se usa el detector congelado de E2 ([YOLO](https://github.com/ultralytics/ultralytics)), con el mismo
preprocesamiento, NMS y umbrales. [ONNX Runtime CUDA](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html)
y el [plugin oficial CUDA 12 v0.3.0](https://github.com/microsoft/onnxruntime/releases/tag/plugin-ep-cuda/v0.3.0)
fijado por SHA256 permiten usar GPU conservando el runtime de la caché CPU.

Antes del lote se comprueba la asignación de convoluciones a CUDA y se comparan hasta 16 gatos y 16 perros,
seleccionados por SHA256 y fuera de las identidades públicas reservadas. Se exige igual número, especie y orden
de detecciones; tolerancias propias: puntuación ≤ 1e-4, coordenadas ≤ 0,1 píxeles.
La muestra debe coincidir y resultar más rápida en GPU; de lo contrario se detiene y guarda el diagnóstico.
Esto **no prueba igualdad en todas las fotos**. `gpu-validation.json` conserva la prueba y sus tiempos.

La limpieza acepta exactamente un animal de la especie esperada; no certifica identidades.
Los splits públicos **val/test v1** se conservan. Los respaldos periódicos verifican bloques en Drive;
una desconexión brusca conserva solo los respaldos confirmados.

Un índice SHA256 en memoria comparte verificaciones entre etapas. Cada archivo se vuelve a leer si cambia
su tamaño, mtime, ctime o identidad en disco. El índice se descarta al finalizar; otra sesión verifica
de nuevo. La referencia v1 completa se carga localmente sin consultar HF cuando coincide con el commit
y fingerprint fijados. Los bloques transferidos desde Drive conservan su verificación completa.


La ampliación conserva las exclusiones manuales, cuarentena de etiquetas contradictorias,
duplicados exactos/cercanos y mínimo de fotos por identidad. Puede cambiar el conjunto retenido
al descubrir conflictos nuevos; las muestras **raw** sí son anidadas.


In [ ]:
started = perf_counter()
prepared = preparation.prepare(detector_dir=os.environ.get("E3_DETECTOR_DIR"))
print("Dataset local:", preparation.version_path)
print("Fingerprint:", prepared.fingerprint)
print(f"Preparación: {(perf_counter() - started) / 60:.1f} min")


## 4. Auditoría antes y después

El reporte conserva cada exclusión, cajas y puntuaciones YOLO, hashes, procedencia y configuración. La cifra de identidades solicitadas es un máximo antes de filtrar, no una cantidad garantizada para entrenar. Las etiquetas pendientes conservan su condición de origen sin verificar.


In [ ]:
audit = json.loads((prepared.root / "audit.json").read_text())
screening = json.loads((prepared.root / "animal-screening.json").read_text())
summary = prepared.summary
print("Fotos candidatas:", summary["candidate_photos"], "→ retenidas:", summary["retained_photos"])
print("Identidades candidatas:", summary["candidate_identities"], "→ retenidas:", summary["retained_identities"])
print("YOLO:", json.dumps(screening["counts"], indent=2))
print("Motivos de exclusión:", json.dumps(summary["exclusion_reasons"], indent=2))
print("Duplicados eliminados:", len(audit["removed_duplicates"]))
print("Identidades en cuarentena:", len(audit["conflicting_identities_quarantined"]))
lines = ["| Especie / split | Identidades | Fotos |", "|---|---:|---:|"]
for split, count in summary["counts"].items():
    lines.append(f"| {split} | {count['identities']} | {count['photos']} |")
display(Markdown("\n".join(lines)))
print(summary["limitation"])

print("Procesamiento por fuente:", json.dumps(summary.get("processing", {}), indent=2))


## 5. Muestra visual reproducible

Se muestran cuatro identidades por especie después del filtrado. La selección queda guardada en `visual-sample.json`; permite inspeccionar problemas visibles sin aprobar automáticamente el resto del dataset.


In [ ]:
sample = json.loads((prepared.root / "visual-sample.json").read_text())
for item in sample["sample"]:
    print(item["animal"])
    display(Image(filename=str(prepared.root / item["sheet"])))


## 6. Publicación opcional en Hugging Face

Drive conserva la versión refinada aunque `PUBLISH_TO_HF=False`. Activarlo publica únicamente
la versión preparada en el repositorio privado, con imágenes e inventario; nunca las fuentes completas.
La versión existente no se sobrescribe. Esta preparación no constituye un experimento ni publica resultados MLflow.


In [ ]:
release = preparation.publish() if PUBLISH_TO_HF else None
print(json.dumps(release, indent=2) if release else "Refined guardada en Drive; publicación HF desactivada.")


## 7. Consumir desde E3

En una sesión nueva, E3 restaura solo `prepared` de los respaldos de Drive y carga el recibo
`v2.20.json`, `v2.50.json` o `v2.100.json`. Verifica hashes antes de entrenar; no ejecuta esta preparación.
Si se usa HF, se fijan commit y fingerprint de la publicación. PetTrace v4 se evalúa por separado.

La optimización conserva la metodología, pero registra una implementación distinta en la procedencia.
Este cuaderno no prueba una aceleración determinada en Colab: depende del disco, Drive y hardware asignado.


In [ ]:
print("E3_PREPARED_DATASET =", preparation.version_path)
if release:
    print(json.dumps({
        "HF_TRAINING_DATASET_REPO": release["repo"],
        "HF_TRAINING_DATASET_VERSION": release["version"],
        "HF_TRAINING_DATASET_REVISION": release["commit"],
        "HF_TRAINING_DATASET_FINGERPRINT": release["fingerprint"],
    }, indent=2))

print("Versión refinada lista:", preparation.version, "· fingerprint:", prepared.fingerprint)
